# **1.Dependencies & Imports**

In [0]:

import json
from datetime import datetime, timezone
from typing import Optional

from pyspark.sql import functions as F
from pyspark.sql import types as T

# **2.Widgets (Parameters)**

In [0]:
%skip
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("bronze_schema", "bronze_market")
dbutils.widgets.text("silver_schema", "silver_market")

dbutils.widgets.text("bronze_table", "crypto_ohlc_raw")
dbutils.widgets.text("silver_table", "crypto_ohlc_1d")
dbutils.widgets.text("silver_table", "") 
dbutils.widgets.text("interval", "1d")

INTERVAL = dbutils.widgets.get("interval").strip()
SILVER_TABLE_IN = dbutils.widgets.get("silver_table").strip()

if not SILVER_TABLE_IN:
    SILVER_TABLE = f"crypto_ohlc_{INTERVAL}"         # 1d -> crypto_ohlc_1d
else:
    SILVER_TABLE = SILVER_TABLE_IN

dbutils.widgets.text("source", "coinbase")      # filter source (optional)
dbutils.widgets.text("symbols", "")            # comma-separated, optional
dbutils.widgets.text("start_date", "")         # YYYY-MM-DD, optional (event_time filter)
dbutils.widgets.text("end_date", "")           # YYYY-MM-DD, optional (event_time filter)

CATALOG = dbutils.widgets.get("catalog").strip()
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema").strip()
SILVER_SCHEMA = dbutils.widgets.get("silver_schema").strip()
BRONZE_TABLE = dbutils.widgets.get("bronze_table").strip()
SILVER_TABLE = dbutils.widgets.get("silver_table").strip()

SOURCE = dbutils.widgets.get("source").strip()
SYMBOLS_RAW = dbutils.widgets.get("symbols").strip()
SYMBOLS = [s.strip().upper() for s in SYMBOLS_RAW.split(",") if s.strip()] if SYMBOLS_RAW else []

START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE = dbutils.widgets.get("end_date").strip()

BRONZE_TBL = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
SILVER_TBL = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

print(f"[INFO] Bronze: {BRONZE_TBL}")
print(f"[INFO] Silver: {SILVER_TBL}")

In [0]:
# 2.Widgets (Parameters)
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("bronze_schema", "bronze_market")
dbutils.widgets.text("silver_schema", "silver_market")

dbutils.widgets.text("bronze_table", "crypto_ohlc_raw")

# 新增：interval（决定 Silver 输出粒度与默认表名）
dbutils.widgets.text("interval", "1d")  # 1d / 1m / 1h

# silver_table 可留空：留空则自动 crypto_ohlc_{interval}
dbutils.widgets.text("silver_table", "")

dbutils.widgets.text("source", "coinbase")  # optional
dbutils.widgets.text("symbols", "")         # optional
dbutils.widgets.text("start_date", "")      # optional
dbutils.widgets.text("end_date", "")        # optional

CATALOG = dbutils.widgets.get("catalog").strip()
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema").strip()
SILVER_SCHEMA = dbutils.widgets.get("silver_schema").strip()
BRONZE_TABLE = dbutils.widgets.get("bronze_table").strip()

INTERVAL = dbutils.widgets.get("interval").strip()
SILVER_TABLE_IN = dbutils.widgets.get("silver_table").strip()

SILVER_TABLE = SILVER_TABLE_IN if SILVER_TABLE_IN else f"crypto_ohlc_{INTERVAL}"

SOURCE = dbutils.widgets.get("source").strip()
SYMBOLS_RAW = dbutils.widgets.get("symbols").strip()
SYMBOLS = [s.strip().upper() for s in SYMBOLS_RAW.split(",") if s.strip()] if SYMBOLS_RAW else []

START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE = dbutils.widgets.get("end_date").strip()

BRONZE_TBL = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
SILVER_TBL = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

print(f"[INFO] Bronze: {BRONZE_TBL}")
print(f"[INFO] Silver: {SILVER_TBL}")
print(f"[INFO] interval={INTERVAL}, source={SOURCE}, symbols={SYMBOLS}")

# **3.Read Bronze with filters**

In [0]:
bronze = spark.table(BRONZE_TBL)
# optional filters
if SOURCE:
    bronze = bronze.where(F.col("source") == F.lit(SOURCE))

if SYMBOLS:
    bronze = bronze.where(F.col("symbol").isin(SYMBOLS))

# event_time filter: uses event_time (day anchor) from Bronze
if START_DATE:
    bronze = bronze.where(F.col("event_time") >= F.to_timestamp(F.lit(f"{START_DATE} 00:00:00")))
if END_DATE:
    bronze = bronze.where(F.col("event_time") <= F.to_timestamp(F.lit(f"{END_DATE} 23:59:59")))

# **4.Parse raw_json into "klines" array**

In [0]:
%skip
payload_schema = T.StructType([
    T.StructField("symbol", T.StringType(), True),
    T.StructField("interval", T.StringType(), True),
    T.StructField("klines", T.ArrayType(T.ArrayType(T.StringType())), True)
])

# 解析 JSON
parsed = (bronze
    .withColumn("payload", F.from_json(F.col("raw_json"), payload_schema))
    .withColumn("klines", F.col("payload.klines"))
)

# 炸裂数组 (Explode)
k = (parsed
     .select(
         F.col("source"),
         F.col("symbol"),
         F.col("ingestion_ts"),
         F.explode_outer("klines").alias("kline")
     )
     # Coinbase 数组长度固定为 6，过滤掉任何不完整的数据
     .filter(F.size(F.col("kline")) >= 6)
)

# 解析列 (专为 Coinbase 定制)
# Coinbase 官方 API 顺序: [0:Time, 1:Low, 2:High, 3:Open, 4:Close, 5:Volume]
k2 = (k
  # 1. 时间戳处理 (Coinbase 是秒级时间戳，直接转换，无需除以1000)
  .withColumn("bar_start_ts", F.col("kline").getItem(0).cast("long").cast("timestamp"))
  
  # 2. OHLCV 数值映射
  .withColumn("low",    F.col("kline").getItem(1).cast("double"))  # Index 1 = Low
  .withColumn("high",   F.col("kline").getItem(2).cast("double"))  # Index 2 = High
  .withColumn("open",   F.col("kline").getItem(3).cast("double"))  # Index 3 = Open
  .withColumn("close",  F.col("kline").getItem(4).cast("double"))  # Index 4 = Close
  .withColumn("volume", F.col("kline").getItem(5).cast("double"))  # Index 5 = Volume

  # 3. 结束时间计算
  # Coinbase 不提供结束时间，根据 Start Time + 1分钟 自动生成
  .withColumn("bar_end_ts", F.col("bar_start_ts") + F.expr("INTERVAL 1 MINUTE"))
)

# 最终选择列
silver_df = (k2
  .withColumn("p_date", F.to_date("bar_start_ts"))
  .select(
      "source", "symbol", "bar_start_ts", "bar_end_ts",
      "open", "high", "low", "close", "volume",
      "ingestion_ts", "p_date"
  )
)

In [0]:


# =========================================================
# 1. 准备工作：定义时间单位映射
# =========================================================
# 假设你在前面的 Cell 已经通过 dbutils 获取了 INTERVAL 变量 (例如 "1d")
# 如果没有，请取消下面这行的注释手动指定，或者确保 widgets 已运行
# INTERVAL = dbutils.widgets.get("interval") 

interval_mapping = {
    "1m": "1 MINUTE",
    "1h": "1 HOUR",
    "1d": "1 DAY"
}
# 获取 SQL 用的时间间隔字符串，默认为 1 MINUTE 以防万一
sql_interval = interval_mapping.get(INTERVAL, "1 MINUTE")

print(f"当前处理的时间间隔: {INTERVAL}, SQL 计算将使用: INTERVAL {sql_interval}")

# =========================================================
# 2. 读取 Bronze 数据 (建议加上 Filter 防止读取旧的脏数据)
# =========================================================
# 确保只处理当前 interval 的数据，避免混入之前跑错的 1m 数据
bronze = spark.read.table("enterprise.bronze_market.crypto_ohlc_raw") \
    .filter(F.col("interval") == INTERVAL)

# =========================================================
# 3. 定义 Schema (关键修改：使用 DoubleType)
# =========================================================
payload_schema = T.StructType([
    T.StructField("symbol", T.StringType(), True),
    T.StructField("interval", T.StringType(), True),
    # [修改点]: 这里必须是 DoubleType，因为 JSON 里的数字没有双引号
    T.StructField("klines", T.ArrayType(T.ArrayType(T.DoubleType())), True) 
])

# =========================================================
# 4. 解析与转换逻辑
# =========================================================

# 解析 JSON
parsed = (bronze
    .withColumn("payload", F.from_json(F.col("raw_json"), payload_schema))
    .withColumn("klines", F.col("payload.klines"))
)

# 炸裂数组 (Explode)
k = (parsed
     .select(
         F.col("source"),
         F.col("symbol"),
         F.col("ingestion_ts"),
         F.explode_outer("klines").alias("kline")
     )
     # Coinbase 数组长度固定为 6，过滤掉任何不完整的数据
     .filter(F.size(F.col("kline")) >= 6)
)

# 解析列 (专为 Coinbase 定制)
# Coinbase 官方 API 顺序: [0:Time, 1:Low, 2:High, 3:Open, 4:Close, 5:Volume]
k2 = (k
  # 1. 时间戳处理 
  # kline[0] 是数字，先转 long 再转 timestamp
  .withColumn("bar_start_ts", F.col("kline").getItem(0).cast("long").cast("timestamp"))
  
  # 2. OHLCV 数值映射
  .withColumn("low",    F.col("kline").getItem(1).cast("double"))  # Index 1 = Low
  .withColumn("high",   F.col("kline").getItem(2).cast("double"))  # Index 2 = High
  .withColumn("open",   F.col("kline").getItem(3).cast("double"))  # Index 3 = Open
  .withColumn("close",  F.col("kline").getItem(4).cast("double"))  # Index 4 = Close
  .withColumn("volume", F.col("kline").getItem(5).cast("double"))  # Index 5 = Volume

  # 3. 结束时间计算 (关键修改：动态计算)
  # 使用 f-string 将 sql_interval (例如 "1 DAY") 注入到 SQL 表达式中
  .withColumn("bar_end_ts", F.col("bar_start_ts") + F.expr(f"INTERVAL {sql_interval}"))
)

# 最终选择列
silver_df = (k2
  .withColumn("p_date", F.to_date("bar_start_ts"))
  .select(
      "source", "symbol", "bar_start_ts", "bar_end_ts",
      "open", "high", "low", "close", "volume",
      "ingestion_ts", "p_date"
  )
)

# 打印结果检查
print(f"Silver DataFrame 生成行数: {silver_df.count()}")
display(silver_df)

# **5. Data quality filters (basic, safe)**


In [0]:
silver_df = (silver_df
  .where(F.col("bar_start_ts").isNotNull())
  .where(F.col("open")  > 0)
  .where(F.col("high")  > 0)
  .where(F.col("low")   > 0)
  .where(F.col("close") > 0)
  .where(F.col("volume") >= 0)
  .where(F.col("high") >= F.greatest(F.col("open"), F.col("close")))
  .where(F.col("low")  <= F.least(F.col("open"), F.col("close")))
)

# intra-batch dedupe
silver_df = silver_df.dropDuplicates(["source","symbol","bar_start_ts"])

print(f"[INFO] Parsed rows (after quality+dedupe): {silver_df.count()}")

# **6.Ensure Silver table exists (if not created yet)**

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TBL} (
  source        STRING,
  symbol        STRING,
  interval      STRING, 
  bar_start_ts  TIMESTAMP,
  bar_end_ts    TIMESTAMP,
  open          DOUBLE,
  high          DOUBLE,
  low           DOUBLE,
  close         DOUBLE,
  volume        DOUBLE,
  ingestion_ts  TIMESTAMP,
  p_date        DATE
)
USING DELTA
PARTITIONED BY (p_date)
""")



# 7. MERGE into Silver (dedupe by source+symbol+bar_start_ts)

In [0]:
stg_view = f"stg_ohlc_{INTERVAL}"
silver_df = silver_df.withColumn("interval", F.lit(INTERVAL))
silver_df.createOrReplaceTempView(stg_view)

spark.sql(f"""
MERGE INTO {SILVER_TBL} AS tgt
USING {stg_view} AS src
ON  tgt.source = src.source
AND tgt.symbol = src.symbol
AND tgt.interval = src.interval
AND tgt.bar_start_ts = src.bar_start_ts
WHEN MATCHED THEN UPDATE SET
  tgt.bar_end_ts     = src.bar_end_ts,
  tgt.open           = src.open,
  tgt.high           = src.high,
  tgt.low            = src.low,
  tgt.close          = src.close,
  tgt.volume         = src.volume,
  tgt.ingestion_ts   = src.ingestion_ts,
  tgt.p_date         = src.p_date,
  tgt.interval       = src.interval
WHEN NOT MATCHED THEN INSERT *
""")


print("[INFO] MERGE completed.")



# **8.Sanity checks**


In [0]:
display(
    spark.table(SILVER_TBL)
         .orderBy(F.col("bar_start_ts").desc())
         .limit(500)
)